# fibonacci_finite_size_scaling
## 1. 研究目的
同じ初期seed pairを固定し、`1 → 10, 0 → 1` の完全なinflationだけで系サイズを増やします。
固定長の有限領域W内で連続消失と任意消失を比較し、有限サイズ依存性を調べます。

**最初の「Run All」は準備・benchmark・validationまでです。**
本計算は、設定セルの `RUN_FULL_ANALYSIS = True` に変更して再実行します。
推定時間が上限を超える場合も本計算を開始しません。

同じフォルダのPythonファイルと `reference/` が必要です。配布ZIPを展開し、このNotebookを開いてください。
実行環境: Python 3.10以上、numpy、pandas、matplotlib、Jupyter。

## 2. これまでの結果と定義の継承
従来の研究では連続消失への強さが観測されました。一方、任意消失はring全体から位置を選んでいました。
今回は消失可能領域を固定するため、従来とは異なる消失配置の集合を調べます。

`fibonacci_recoverability_kl` の `recoverability_kl.py` と `legacy_functions.py` を**変更せず同梱**しています。
既存の `remaining_pattern`, `recoverability`, collision集計、`reduced_operator`、KL対角・非対角判定を使います。
論理状態は**重複を除いた**巡回軌道の正規化された等重ね合わせです。両軌道サイズが違っても、それぞれのサイズで正規化します。

再利用元とSHA256は `reference/` と `source_manifest.json` に記録しています。

## 3. finite-size scalingの考え方
固定するものはseed pair、W、m、相対消失パターンです。変えるものはinflation回数nです。
`N = len(inflate(SEED_0, n))` を自動取得し、途中でbit列を切り取りません。

windowは表示上の中央に置きますが、物理系はringです。中央配置はopen boundaryから遠ざける操作ではありません。
連続消失はwindow内の `W-m+1` 通りを全探索します。translation invarianceが成立すれば、固定pair,n,mでの連続消失率は0または1になります。

window全体がKL訂正可能なら、その全ての部分集合も訂正可能です。この包含関係も確認します。
単一サイズを無限系とは呼びません。1/N図に根拠のない外挿fitは加えません。

## 4. 手動入力セル
==============================
**ここはユーザーが手動で入力する**
==============================

--------------------------------------------------
**ここから下はユーザーが手動で設定する項目**
--------------------------------------------------

通常はこの1セルだけ変更してください。既定seedは過去の研究資料にある `L06_W03_P0001` です。
追加pairは `EXTRA_SEED_PAIRS` に辞書を追加できます。各pairは独立したlogical bitとして解析し、多次元コードを構成するものではありません。

In [1]:
# ==============================
# 手動入力する解析パラメータ
# ==============================
SEED_0 = "000111"
SEED_1 = "010101"
PAIR_ID = "L06_W03_P0001"
EXTRA_SEED_PAIRS = []  # 例: [{"pair_id": "pair_2", "seed_0": "...", "seed_1": "..."}]

INFLATION_LIST = [6, 7, 8, 9, 10]
WINDOW_SIZE_LIST = [20, 30, 50]
ERASURE_SIZE_LIST = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
MAX_EXACT_PATTERNS = 100000
NUM_RANDOM_SAMPLES = 10000
RANDOM_SEED = 42

SAVE_RESULTS = True
SAVE_FIGURES = True
OUTPUT_DIR = "results"

# detailは逐次保存し、画面表示用のメモリ保持はこの上限まで。
DETAIL_MODE = "all"  # all / failures / representative / none
DETAIL_MEMORY_LIMIT = 2000
DETAIL_REPRESENTATIVE_LIMIT = 100  # representative時: 全体で残す先頭の件数

BENCHMARK_PATTERNS = 100  # 各pair,nにつき100～1000。候補が少ない場合は同じ候補を繰り返し測定。
MAX_ESTIMATED_SECONDS = 600  # benchmarkを確認してから必要なら変更
RUN_FULL_ANALYSIS = False  # benchmarkとvalidationを確認後、Trueにして再実行

--------------------------------------------------
**ここから下は基本的に自動計算**
--------------------------------------------------

N、orbit、windowの位置、絶対消失位置、訂正率、理論長は自動生成します。

In [2]:
from pathlib import Path
import sys
import json
import hashlib
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "finite_size_scaling.py").exists():
    raise FileNotFoundError("ZIPを展開し、Notebookと同じフォルダを作業ディレクトリにしてください。")
sys.path.insert(0, str(BASE_DIR))
import recoverability_kl as kl
import finite_size_scaling as fs
from theory import theory_table, paper_seed_conditions
from scaling_plots import plot_scaling, plot_heatmaps
from reporting import convergence_assessment, conclusion_markdown

CONFIG_KEYS = ["SEED_0", "SEED_1", "PAIR_ID", "EXTRA_SEED_PAIRS", "INFLATION_LIST",
    "WINDOW_SIZE_LIST", "ERASURE_SIZE_LIST", "MAX_EXACT_PATTERNS", "NUM_RANDOM_SAMPLES",
    "RANDOM_SEED", "SAVE_RESULTS", "SAVE_FIGURES", "OUTPUT_DIR", "DETAIL_MODE",
    "DETAIL_MEMORY_LIMIT", "DETAIL_REPRESENTATIVE_LIMIT", "BENCHMARK_PATTERNS",
    "MAX_ESTIMATED_SECONDS", "RUN_FULL_ANALYSIS"]
config = {key: globals()[key] for key in CONFIG_KEYS}
result = None
summary_df = pd.DataFrame()
scaling_df = pd.DataFrame()
detail_df = pd.DataFrame()
theory_comparison_df = pd.DataFrame()

## 5. 入力条件の検証
非空のbinary seed、同じk0/k1、巡回非同値、両方向の1回隣接交換条件を確認します。
位置は左から0始まり、ring境界の隣接交換も含みます。
`N < W` と `m > W` の組は警告してスキップします。Wやmが正整数でない等の設定ミスは停止します。

In [3]:
kl.verify_reference(BASE_DIR / "reference")
source_manifest = json.loads((BASE_DIR / "source_manifest.json").read_text())
for filename, expected in source_manifest["reused_files"].items():
    actual = hashlib.sha256((BASE_DIR / filename).read_bytes()).hexdigest()
    if actual != expected["sha256"]:
        raise RuntimeError(f"既存判定関数のhash不一致: {filename}")
prepared = fs.prepare(config)
print("入力検証・既存関数の同一性確認: 合格")

入力検証・既存関数の同一性確認: 合格


## 6. inflationによるNの自動生成
`prepare` はpair,nごとに `F0_n = inflate(seed_0,n)` と `F1_n = inflate(seed_1,n)` を生成し、同長性と支持の非交差を確認します。
各orbitは一度だけ生成し、W,mごとには生成し直しません。2^Nのstatevectorやdensity matrixは使用しません。

In [4]:
for (pair_id, n), model in prepared["models"].items():
    assert model.N == len(model.F0) == len(model.F1)
    assert model.F0 == kl.inflate(model.seed_0, n)
    assert model.F1 == kl.inflate(model.seed_1, n)

## 7. n–N対応表
以降の本計算より先に、実際に解析するサイズを表示します。

In [5]:
size_df = prepared["size_table"]
display(size_df)

,pair_id,seed_length,k0,k1,adjacent_swap_valid,seed_0,seed_1,inflation_n,N,orbit_size_0,orbit_size_1
0,L06_W03_P0001,6,3,3,True,000111,010101,6,102,102,34
1,L06_W03_P0001,6,3,3,True,000111,010101,7,165,165,55
2,L06_W03_P0001,6,3,3,True,000111,010101,8,267,267,89
3,L06_W03_P0001,6,3,3,True,000111,010101,9,432,432,144
4,L06_W03_P0001,6,3,3,True,000111,010101,10,699,699,233


## 8. logical pairの隣接交換条件チェック
`adjacent_swap_valid` がユーザー指定の条件です。`paper_seed_condition` は理論保証の適用に必要な条件を別途示します。
後者には自己巡回同値類へ戻る隣接交換の排除も含まれます。理論保証の追加条件によって入力を拒否することはありません。

In [6]:
seed_checks_df = pd.DataFrame([
    dict(pair, **paper_seed_conditions(pair["seed_0"], pair["seed_1"]))
    for pair in prepared["pairs"]
])
display(seed_checks_df)
assert seed_checks_df.adjacent_swap_valid.all()

,pair_id,seed_0,seed_1,seed_length,k0,k1,adjacent_swap_valid,same_composition,cyclic_inequivalent,cross_swap_01,cross_swap_10,self_swap_0,self_swap_1,paper_seed_condition
0,L06_W03_P0001,000111,010101,6,3,3,True,True,True,False,False,False,False,True


## 9. cyclic orbit生成
重複を除いたO0,O1を再利用します。周期seedでは `orbit_size < N` です。正規化をNに置き換えません。

In [7]:
orbit_df = pd.DataFrame([
    {"pair_id": pair_id, "inflation_n": n, "N": model.N,
     "orbit_size_0": len(model.O0), "orbit_size_1": len(model.O1)}
    for (pair_id, n), model in prepared["models"].items()
])
display(orbit_df)

,pair_id,inflation_n,N,orbit_size_0,orbit_size_1
0,L06_W03_P0001,6,102,102,34
1,L06_W03_P0001,7,165,165,55
2,L06_W03_P0001,8,267,267,89
3,L06_W03_P0001,9,432,432,144
4,L06_W03_P0001,10,699,699,233


## 10. window生成
`window_start=(N-W)//2`、`window_end=window_start+W-1`（終端を含む）です。
相対位置 `r` は通常 `window_start+r` に変換します。translation testではmod Nを用いて表現の切れ目をまたぐ位置も試します。

In [8]:
windows_df = prepared["windows"]
display(windows_df)

,pair_id,seed_0,seed_1,inflation_n,N,window_size,window_start,window_end,window_positions_absolute
0,L06_W03_P0001,000111,010101,6,102,20,41,60,"(41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5..."
1,L06_W03_P0001,000111,010101,6,102,30,36,65,"(36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4..."
2,L06_W03_P0001,000111,010101,6,102,50,26,75,"(26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 3..."
3,L06_W03_P0001,000111,010101,7,165,20,72,91,"(72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 8..."
4,L06_W03_P0001,000111,010101,7,165,30,67,96,"(67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 7..."
5,L06_W03_P0001,000111,010101,7,165,50,57,106,"(57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 6..."
6,L06_W03_P0001,000111,010101,8,267,20,123,142,"(123, 124, 125, 126, 127, 128, 129, 130, 131, ..."
7,L06_W03_P0001,000111,010101,8,267,30,118,147,"(118, 119, 120, 121, 122, 123, 124, 125, 126, ..."
8,L06_W03_P0001,000111,010101,8,267,50,108,157,"(108, 109, 110, 111, 112, 113, 114, 115, 116, ..."
9,L06_W03_P0001,000111,010101,9,432,20,206,225,"(206, 207, 208, 209, 210, 211, 212, 213, 214, ..."


## 11. translation invariance test
同じ形の消失集合を複数のwindow開始位置へ移動させ、KL対角・非対角・correctabilityとrecoverabilityを照合します。
不一致時は本計算を停止します。

In [9]:
translation_df = fs.translation_test(prepared)
display(translation_df)

,pair_id,inflation_n,N,window_size,translated_start,erasure_positions_relative,erasure_positions_absolute,KL_correctable,matched
0,L06_W03_P0001,6,102,20,0,"(0,)","(0,)",True,True
1,L06_W03_P0001,6,102,20,41,"(0,)","(41,)",True,True
2,L06_W03_P0001,6,102,20,51,"(0,)","(51,)",True,True
3,L06_W03_P0001,6,102,20,101,"(0,)","(101,)",True,True
4,L06_W03_P0001,6,102,20,0,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",True,True
...,...,...,...,...,...,...,...,...,...
175,L06_W03_P0001,10,699,50,698,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",True,True
176,L06_W03_P0001,10,699,50,0,"(0, 25, 49)","(0, 25, 49)",True,True
177,L06_W03_P0001,10,699,50,324,"(0, 25, 49)","(324, 349, 373)",True,True
178,L06_W03_P0001,10,699,50,349,"(0, 25, 49)","(349, 374, 398)",True,True


## 12. erasure pattern生成
`C(W,m) <= MAX_EXACT_PATTERNS` なら全探索、それ以外は独立一様な復元抽出です。
同じパターンが抽出された試行も保持し、試行数として数えます。これはWilson区間の二項試行モデルと対応します。

パターン生成の乱数にはNやpairを含めません。同じW,mでは同じseedから同じ順序の相対パターン列を再生成します。
`pattern_index` とdigestで共有を確認できます。全パターンをメモリに置く必要はありません。

In [10]:
pattern_plan_df = pd.DataFrame([
    {"window_size": W, "erasure_size": m,
     "total_possible_patterns": plan.total_patterns,
     "number_of_patterns": plan.sample_count,
     "calculation_mode": "exact" if plan.mode == "exact" else "sampled",
     "contiguous_patterns": W - m + 1}
    for (W, m), plan in sorted(prepared["plans"].items())
])
display(pattern_plan_df)

,window_size,erasure_size,total_possible_patterns,number_of_patterns,calculation_mode,contiguous_patterns
0,20,1,20,20,exact,20
1,20,2,190,190,exact,19
2,20,3,1140,1140,exact,18
3,20,4,4845,4845,exact,17
4,20,5,15504,15504,exact,16
5,20,6,38760,38760,exact,15
6,20,7,77520,77520,exact,14
7,20,8,125970,10000,sampled,13
8,20,9,167960,10000,sampled,12
9,20,10,184756,10000,sampled,11


## 13. benchmark
各pair,nで100～1000試行を上限に評価し、orbitサイズ、実測時間、1パターンの平均時間、10,000パターンの推定時間を表示します。
W,mごとの重さが違うため複数の組を測ります。推定値は計算本体の目安であり、保存・表示等の時間も必要です。

推定時間が上限を超えた場合は自動で本計算しません。サンプル数・inflation回数・W,mを減らすか、見積りを確認して時間上限を変更してください。

In [11]:
benchmark_df = fs.benchmark(prepared, config)
display(benchmark_df)
if not benchmark_df.empty:
    estimated_seconds = float(benchmark_df.estimated_full_seconds.sum())
    print(f"本計算の推定合計: {estimated_seconds:,.1f}秒 ({estimated_seconds / 60:.1f}分)")
    display(benchmark_df.groupby(["pair_id", "inflation_n", "N", "orbit_size_0", "orbit_size_1"], as_index=False).agg(patterns=("num_benchmark_patterns", "sum"), seconds=("elapsed_seconds", "sum"), estimated_full_seconds=("estimated_full_seconds", "sum")))

本計算の推定合計: 911.1秒 (15.2分)


/Users/ideryuito/Documents/Codex/2026-10-06/files-pasted-by-the-user-fibonacci-3/outputs/fibonacci_finite_size_scaling/finite_size_scaling.py:266: RuntimeWarning: 推定 911.1 秒 > 上限 600.0 秒。設定を縮小するかbenchmarkを確認して上限を変更してください。本計算は未実行です。
  warnings.warn(f'推定 {estimated:.1f} 秒 > 上限 {c["MAX_ESTIMATED_SECONDS"]:.1f} 秒。'


,pair_id,inflation_n,N,window_size,erasure_size,orbit_size_0,orbit_size_1,num_benchmark_patterns,elapsed_seconds,seconds_per_pattern,timing_source,planned_arbitrary_patterns,planned_contiguous_patterns,calculation_mode,config_fingerprint,benchmark_safety_factor,estimated_seconds_per_10000,full_window_seconds,estimated_full_seconds
0,L06_W03_P0001,6,102,20,1,102,34,4,0.000334,0.000084,measured,20,20,exact,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,0.835835,0.000146,0.005037
1,L06_W03_P0001,6,102,20,2,102,34,4,0.000327,0.000082,measured,190,19,exact,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,0.817710,0.000146,0.025657
2,L06_W03_P0001,6,102,20,3,102,34,4,0.000339,0.000085,measured,1140,18,exact,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,0.848437,0.000146,0.147395
3,L06_W03_P0001,6,102,20,4,102,34,4,0.000340,0.000085,measured,4845,17,exact,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,0.851042,0.000146,0.620687
4,L06_W03_P0001,6,102,20,5,102,34,4,0.000344,0.000086,measured,15504,16,exact,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,0.860000,0.000146,2.002102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,L06_W03_P0001,10,699,50,6,699,233,3,0.002504,0.000835,measured,10000,45,sampled,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,8.347500,0.000853,12.577723
146,L06_W03_P0001,10,699,50,7,699,233,3,0.002696,0.000899,measured,10000,44,sampled,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,8.987917,0.000853,13.541323
147,L06_W03_P0001,10,699,50,8,699,233,3,0.002476,0.000825,measured,10000,43,sampled,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,8.254584,0.000853,12.435245
148,L06_W03_P0001,10,699,50,9,699,233,3,0.002492,0.000831,measured,10000,42,sampled,952c4b404876196fcee8f8ca318e7b258335f6ff31abce...,1.5,8.306250,0.000853,12.511832


,pair_id,inflation_n,N,orbit_size_0,orbit_size_1,patterns,seconds,estimated_full_seconds
0,L06_W03_P0001,6,102,102,34,100,0.009797,51.600178
1,L06_W03_P0001,7,165,165,55,100,0.014405,74.045784
2,L06_W03_P0001,8,267,267,89,100,0.023567,123.703000
3,L06_W03_P0001,9,432,432,144,100,0.042732,220.875973
4,L06_W03_P0001,10,699,699,233,100,0.082369,440.858977


## 14. 小規模validation
小さいNで全ての消失部分集合を列挙し、既存の整数カウント版と既存Notebookから抽出した疎KL残差版を同じseed,N,Eで照合します。
statevectorは作りません。大きな手動seedでこの全探索が難しい場合は既知の小さいseed pairも使います。
境界・正規化・包含単調性は同梱の `test_finite_size_scaling.py` でも検証します。

In [12]:
validation_df = fs.validate_small(prepared)
display(validation_df)
validation_passed = True  # 不一致時はvalidate_smallが例外を発生させ、ここには到達しない。

,pair_id,seed_0,seed_1,inflation_n,N,erasure_size,erasure_positions,KL_correctable,legacy_KL_correctable,matched
0,L06_W03_P0001,000111,010101,0,6,0,(),True,True,True
1,L06_W03_P0001,000111,010101,0,6,1,"(0,)",True,True,True
2,L06_W03_P0001,000111,010101,0,6,1,"(1,)",True,True,True
3,L06_W03_P0001,000111,010101,0,6,1,"(2,)",True,True,True
4,L06_W03_P0001,000111,010101,0,6,1,"(3,)",True,True,True
...,...,...,...,...,...,...,...,...,...,...
59,L06_W03_P0001,000111,010101,0,6,5,"(0, 1, 2, 4, 5)",False,False,True
60,L06_W03_P0001,000111,010101,0,6,5,"(0, 1, 3, 4, 5)",False,False,True
61,L06_W03_P0001,000111,010101,0,6,5,"(0, 2, 3, 4, 5)",False,False,True
62,L06_W03_P0001,000111,010101,0,6,5,"(1, 2, 3, 4, 5)",False,False,True


## 15. recoverability計算
補領域への射影が各orbit内で一対一かを調べます。
`recoverable_both` は両orbitの自己collisionが0であることです。
これは補助指標であり、量子訂正可能性の代わりには使いません。

`kl.recoverability(orbit,E)` と、既存の `LogicalPair.evaluate(E)` 内のcollision集計を再利用します。
bit mask、補領域grouping、collision counting、reduced-operator countingは独立した既存関数であり、後でC実装に置き換えやすい構造です。

## 16. KL計算
消失領域Eについて
\[
\rho_E^{ij}=\operatorname{Tr}_{E^c}(|i_L\rangle\langle j_L|)
\]
とすると、`KL_diagonal` は \(\rho_E^{00}=\rho_E^{11}\)、`KL_offdiagonal` は \(\rho_E^{01}=0\) です。
両者を満たす場合だけ `KL_correctable=True` です。

補領域が同じorbit語をgroupingし、整数カウントとorbitサイズの交差積で対角条件を厳密に判定します。
正の等振幅なので、交差ブロックの消失はcross collisionが0であることと一致します。
本計算は変更していない `kl.LogicalPair.evaluate(E)` を呼び出します。

## 17. finite-size本計算
`RUN_FULL_ANALYSIS=False` の間はこのセル以降の結果表・グラフは未計算です。
Trueでもbenchmark、translation test、small validationが完了し、時間上限内に収まる必要があります。

各Wの全消失を別にKL判定し、成功する場合に部分集合で失敗しないことを確認します。
集計は全試行に対して逐次行い、detailの保存モードには依存しません。

In [13]:
if RUN_FULL_ANALYSIS:
    result = fs.run_analysis(prepared, config, benchmark_df, validation_passed=validation_passed)
    detail_df = result["detail"]
    summary_df = result["summary"]
    scaling_df = result["scaling"]
    print("本計算が完了しました。")
else:
    print("準備完了。本計算は未実行です。benchmarkを確認後、冒頭のRUN_FULL_ANALYSISをTrueにして再実行してください。")

準備完了。本計算は未実行です。benchmarkを確認後、冒頭のRUN_FULL_ANALYSISをTrueにして再実行してください。


## 18. detail DataFrame
1試行につき1行です。`erasure_positions_relative` と `erasure_positions_absolute` を保存します。
`detail_df` は最大 `DETAIL_MEMORY_LIMIT` 行の表示用抜粋です。`DETAIL_MODE="all"` と `SAVE_RESULTS=True` なら完全なdetailをファイルへ逐次保存します。
failures/representative/noneの場合、その保存範囲をmetadataに記録します。画面の抜粋から訂正率を再計算しないでください。

In [14]:
display(detail_df.head(20))
if result is not None:
    display(result.get("paths", {}))

""


## 19. summary / scaling DataFrame
`gap_KL = R_KL_contiguous - R_KL_arbitrary`、`arbitrary_failure_rate = 1-R_KL_arbitrary` です。
exactにはsampling誤差がありません。sampledには点ごとの95% Wilson区間を付け、全成功でも区間幅を0にはしません。
`number_of_patterns` は両種類の試行数の合計です。`num_patterns_arbitrary` と `num_patterns_contiguous` も別列に保存します。
隣接サイズ間の `Delta_R` は有限サイズの記述量です。単調な減少や無限系への収束を仮定しません。

In [15]:
display(summary_df)
display(scaling_df)

""


""


## 20. correction rate vs N
同じpair,W,mで2種類のKL訂正率を比較します。丸印はexact、菱形はsampledです。error barはsampledの95%区間です。
複数pairの平均に置き換えず、それぞれの系列を描きます。

In [16]:
for fig in plot_scaling(summary_df, "rates", Path(OUTPUT_DIR) / "figures", SAVE_FIGURES):
    display(fig)
    plt.close(fig)

## 21. gap / arbitrary failure / 1/N
固定W内の任意消失は連続消失パターンも含みますが、2種類の率は異なる集合の平均なのでgapが負になる可能性もあります。
グラフは負のgapを隠しません。failure rateの0は対数軸ではなく線形軸に表示します。
1/N図にfitは加えません。

In [17]:
for kind in ["gap", "failure", "inverse_N"]:
    for fig in plot_scaling(summary_df, kind, Path(OUTPUT_DIR) / "figures", SAVE_FIGURES):
        display(fig)
        plt.close(fig)

## 22. heatmap
pair,Wごとに、横軸m、縦軸N、色をKL訂正率とした2種類のheatmapを表示します。
`*` はsampledです。スキップした組は補間せず欠損のまま扱います。

In [18]:
for fig in plot_heatmaps(summary_df, Path(OUTPUT_DIR) / "figures", SAVE_FIGURES):
    display(fig)
    plt.close(fig)

## 23. theoretical scaleとの比較
Li & Boyle, *The Penrose Tiling is a Quantum Error-Correcting Code*, arXiv:2311.13040v2,
[Appendix E.1 / E.3](https://arxiv.org/html/2311.13040v2#A5.SS1) を参照します。
原論文の添字は \(f_0=f_1=1\)、保証長は \(L_{\mathrm{theory}}(n)=f_n+1\) です。
式の値と保証の適用を区別し、n=0や原論文のseed条件を満たさない場合は保証と表示しません。

保証外は「訂正不能」ではなく「この十分条件では未判定」です。詳細は `THEORY.md` を参照してください。
理論上保証されたwindowが数値判定で失敗した場合は停止します。

In [19]:
theory_df = theory_table(prepared)
if result is not None:
    join_keys = ["pair_id", "inflation_n", "N", "window_size"]
    actual = result["windows"]
    keep = join_keys + [c for c in actual.columns if c not in theory_df.columns]
    theory_comparison_df = theory_df.merge(actual[keep], on=join_keys, how="left", validate="one_to_one")
    if "KL_correctable_window" in theory_comparison_df:
        contradiction = theory_comparison_df.theory_window_guaranteed & ~theory_comparison_df.KL_correctable_window.fillna(False).astype(bool)
        if contradiction.any():
            display(theory_comparison_df[contradiction])
            raise AssertionError("理論保証とKL実測値が不一致です。添字・seed条件・orbit・indexを確認してください。")
else:
    theory_comparison_df = theory_df
display(theory_comparison_df)

,pair_id,seed_0,seed_1,inflation_n,N,window_size,window_start,window_end,window_positions_absolute,n,W,L_theory,W_le_L_theory,same_composition,cyclic_inequivalent,cross_swap_01,cross_swap_10,self_swap_0,self_swap_1,adjacent_swap_valid,paper_seed_condition,theory_guarantee_applicable,predicted_window_correctable,theory_window_guaranteed,theory_reference
0,L06_W03_P0001,000111,010101,6,102,20,41,60,"(41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...",6,20,14,False,True,True,False,False,False,False,True,True,True,<NA>,False,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
1,L06_W03_P0001,000111,010101,6,102,30,36,65,"(36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...",6,30,14,False,True,True,False,False,False,False,True,True,True,<NA>,False,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
2,L06_W03_P0001,000111,010101,6,102,50,26,75,"(26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 3...",6,50,14,False,True,True,False,False,False,False,True,True,True,<NA>,False,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
3,L06_W03_P0001,000111,010101,7,165,20,72,91,"(72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 8...",7,20,22,True,True,True,False,False,False,False,True,True,True,True,True,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
4,L06_W03_P0001,000111,010101,7,165,30,67,96,"(67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 7...",7,30,22,False,True,True,False,False,False,False,True,True,True,<NA>,False,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
5,L06_W03_P0001,000111,010101,7,165,50,57,106,"(57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 6...",7,50,22,False,True,True,False,False,False,False,True,True,True,<NA>,False,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
6,L06_W03_P0001,000111,010101,8,267,20,123,142,"(123, 124, 125, 126, 127, 128, 129, 130, 131, ...",8,20,35,True,True,True,False,False,False,False,True,True,True,True,True,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
7,L06_W03_P0001,000111,010101,8,267,30,118,147,"(118, 119, 120, 121, 122, 123, 124, 125, 126, ...",8,30,35,True,True,True,False,False,False,False,True,True,True,True,True,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
8,L06_W03_P0001,000111,010101,8,267,50,108,157,"(108, 109, 110, 111, 112, 113, 114, 115, 116, ...",8,50,35,False,True,True,False,False,False,False,True,True,True,<NA>,False,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."
9,L06_W03_P0001,000111,010101,9,432,20,206,225,"(206, 207, 208, 209, 210, 211, 212, 213, 214, ...",9,20,56,True,True,True,False,False,False,False,True,True,True,True,True,"Li & Boyle, arXiv:2311.13040v2, Appendix E.1, ..."


## 24. threshold解析とlogical pair間比較
`n_star`, `N_star` は**調べたサイズ列の中で**初めて成功率1となったサイズです。未計算サイズも含む真の最小値ではありません。
`sampled_observed` は抽出試行の全成功で、全集合の証明ではありません。exact全成功、全windowのKL成功による保証を区別します。
後続サイズの成功も確認し、最初の率1だけで全てのより大きいNを断定しません。

pair間のmean/median/min/maxは補助表です。seedごとの結果を必ず残し、異なるW,mや異なるNを混ぜません。

In [20]:
if result is not None:
    display(result["thresholds"])
    display(result["pair_statistics"])
assessment_df = convergence_assessment(summary_df)
display(assessment_df)

,pair_id,window_size,erasure_size,N_first,N_last,R_first,R_last,observed_change,gap_last,delta_last,observed_nondecreasing,all_sizes_exact,interpretation


## 25. 結果まとめ
数値的な改善はfinite-size effectと整合的ですが、固定Wの結果から従来のring全域の任意消失まで一般化しません。
改善が観測されない場合も、試したサイズ範囲とsampling精度を確認し、より大きいNでの可能性を即座には否定しません。

原論文の条件を満たすpairでは、固定Wが理論保証長に入った後の全subset訂正は理論からも従います。
その前の変化、実測threshold、保証との整合性を本Notebookで調べます。
mutual information、correlation、structure factor、seed最適化、Hamiltonian、2D modelは今回の範囲に含めません。

In [21]:
display(Markdown(conclusion_markdown(prepared, config, result, theory_comparison_df)))
display(size_df)
if not summary_df.empty:
    display(summary_df[["pair_id", "inflation_n", "N", "window_size", "erasure_size",
                        "R_KL_contiguous", "R_KL_arbitrary", "gap_KL", "calculation_mode"]])
if SAVE_RESULTS:
    destination = Path(OUTPUT_DIR)
    destination.mkdir(parents=True, exist_ok=True)
    for name, table in {"size_table": size_df, "seed_checks": seed_checks_df,
                        "pattern_plan": pattern_plan_df, "translation_test": translation_df,
                        "benchmark": benchmark_df, "small_validation": validation_df,
                        "theory_comparison": theory_comparison_df, "convergence_assessment": assessment_df}.items():
        table.to_csv(destination / (name + ".csv"), index=False)
    (destination / "notebook_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2))
    (destination / "conclusions.md").write_text(conclusion_markdown(prepared, config, result, theory_comparison_df))

### 自動まとめ

- `L06_W03_P0001`: seed 0 = `000111` / seed 1 = `010101`。指定の隣接交換条件: 合格。
- INFLATION_LIST = `[6, 7, 8, 9, 10]`
- WINDOW_SIZE_LIST = `[20, 30, 50]`
- ERASURE_SIZE_LIST = `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`

本計算はまだ実行していません。以下のN一覧・benchmark・validationは準備段階の結果です。率や有限サイズ効果の結論は未計算です。

,pair_id,seed_length,k0,k1,adjacent_swap_valid,seed_0,seed_1,inflation_n,N,orbit_size_0,orbit_size_1
0,L06_W03_P0001,6,3,3,True,000111,010101,6,102,102,34
1,L06_W03_P0001,6,3,3,True,000111,010101,7,165,165,55
2,L06_W03_P0001,6,3,3,True,000111,010101,8,267,267,89
3,L06_W03_P0001,6,3,3,True,000111,010101,9,432,432,144
4,L06_W03_P0001,6,3,3,True,000111,010101,10,699,699,233
